In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv("Downloads/Task 3 and 4_Loan_Data.csv")
df.head()

In [ ]:
df.info()

In [ ]:
# Count Default and Non-Default customers
churn_counts = df['default'].value_counts()
labels = ['Non-Default', 'Default']
sizes = churn_counts.values
colors = ['#4CC9F0', '#A52A2A'] 
explode = (0, 0.1)  # Slightly explode the Default slice for emphasis

# Pie chart
plt.figure(figsize=(8, 8), dpi=300)
wedges, texts, autotexts = plt.pie(
    sizes,
    labels=labels,
    autopct='%1.1f%%',
    colors=colors,
    explode=explode,
    startangle=140,
    textprops={'fontsize': 12, 'color': 'black'}  # default text color
)

# Change text color to white for Default section (maroon)
for i, autotext in enumerate(autotexts):
    if labels[i] == 'Default':  # Change text color if it's Default
        autotext.set_color('white')

plt.title('Proportion of Loan Defaults', fontsize=14)
plt.show()

print(df['default'].value_counts())

Feature Engineering and Scaling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Feature Engineering
df['debt_to_income'] = df['total_debt_outstanding'] / df['income']
df['loan_to_income'] = df['loan_amt_outstanding'] / df['income']

X = df.drop(columns=['customer_id', 'default'])
y = df['default']

# Define feature_columns for later (expected loss function)
feature_columns = X.columns.tolist()

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=42, stratify=y)

# Save unscaled test for later (expected loss analysis)
X_test_original = X_test.copy()

# Fit scaler on train only, then transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

Train and Evaluate Logistic Regression
Now we train a Logistic Regression model that outputs probability of default.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# Train the model
model_1 = LogisticRegression(random_state=42)
model_1.fit(X_train_scaled, y_train)

# Predict Predict probabilities on test set
y_proba = model_1.predict_proba(X_test_scaled)[:, 1]

# Predict class
y_pred = model_1.predict(X_test_scaled)

# Evaluate
print("ROC AUC Score:", roc_auc_score(y_test, y_proba))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# # Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5), dpi=300)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Non-Default', 'Default'], 
            yticklabels=['Non-Default', 'Default'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()



Interpretation of Results:
1. Roc AUC Score: 0.999988
Almost perfect. This means the Logistic Regression model is almost perfectly distinguishing between defaulters and non-defaulters.

2. Classification Report
Accuracy: 1996 correct / 2000 total (~100%)
Precision (Default=1): Of predicted defaulters, 100% were correct (1.00)
Recall (Default=1): Caught 367 out of 370 actual defaulters (0.99)
F1-score (Default=1): Balanced measure of precision + recall
3. Confusion Matrix
1629 TN, 1 FP, 3 FN, 367 TP
1629 customers who did not default were correctly predicted as non-default (TN).
1 customer who did not default was incorrectly predicted as default (FP).
3 customers who did default were missed and predicted as non-default (FN).
367 customers who did default were correctly predicted (TP).
Too good to be true? Possibly yes.
Unusual high scores usually mean that either:

the data is too easy for the model to understand (i.e., total_debt_outstanding, income, years_employed, fico_score, etc. might directly correlate with default.
or the dataset has some leakage (a feature that uninentionally "hinting" at the target).
Expected Loss Analysis
Formula: Expected Loss=PD×(1−Recovery Rate)×Loan Amount

PD: Probability of Default predicted by the model.
Recovery Rate: 10% (assumed). Means 90% of loss if default happens.
Loan Amount: loan_amt_outstanding feature.

Expected Loss Analysis:

Formula: Expected Loss=PD×(1−Recovery Rate)×Loan Amount

PD: Probability of Default predicted by the model.
Recovery Rate: 10% (assumed). Means 90% of loss if default happens.
Loan Amount: loan_amt_outstanding feature.

In [ ]:
print(y_proba[:10])


In [ ]:
# Define recovery rate and LGD
recovery_rate = 0.10
lgd = 1 - recovery_rate  # Loss Given Default

# Loan amounts from unscaled test set
loan_amounts = X_test_original['loan_amt_outstanding'].values

# Compute Expected Loss for each customer
expected_loss = y_proba * lgd * loan_amounts

# Add to a new DataFrame for inspection
el_df = X_test_original.copy()
el_df['PD'] = y_proba
el_df['ExpectedLoss'] = expected_loss
el_df['ActualDefault'] = y_test.values

# Show the top 10 customers with the highest expected loss
highest_el = el_df.sort_values(by='ExpectedLoss', ascending=False)
highest_el.head(10)


In [ ]:

df.describe()


1. The model is highly confident
PD values are very close to 1, which suggests that the model is very certain these borrowers will default.
In this top 10 list, every one of them actually did default (ActualDefault = 1).
2. Expected Loss is well-aligned
The formula is working correctly: customers with high loan amounts and high PDs result in large Expected Losses.
3. Credit Quality and Risk Factors
FICO scores are mostly low (534–674), indicating subprime or borderline prime borrowers.
Debt-to-income ratios range from ~24% to 36%, on the higher side.
Years employed is low (1–4), which could affect financial stability.

Expected Loss Function
This is a function that can take in the properties of a loan and output the expected loss.

In [ ]:

import numpy as np

def calculate_expected_loss(borrower_data, model, scaler, feature_columns, recovery_rate=0.10):
    """
    borrower_data: dict with keys matching original features (raw)
    model: trained model
    scaler: fitted scaler used during training
    feature_columns: list of columns used during model training (after feature engineering)
    recovery_rate: float, default 0.10
    """
    # Convert dict to DataFrame
    features = pd.DataFrame([borrower_data])
    
    # Feature engineering
    features['debt_to_income'] = features['total_debt_outstanding'] / features['income']
    features['loan_to_income'] = features['loan_amt_outstanding'] / features['income']

    # Reorder columns to match training
    features = features[feature_columns]

    # Scale
    scaled_features = scaler.transform(features)

    # Predict PD
    pd_default = model.predict_proba(scaled_features)[0][1]

    # Compute Expected Loss
    loan_amount = borrower_data['loan_amt_outstanding']
    expected_loss = pd_default * (1 - recovery_rate) * loan_amount

    return expected_loss

In [ ]:

# Let's use the customer who has the highest Expected Loss (ID 9944)
sample = {
    'credit_lines_outstanding': 4,
    'loan_amt_outstanding': 8989.178801,
    'total_debt_outstanding': 30103.66256,
    'income': 124197.6337,
    'years_employed': 2,
    'fico_score': 641
}

el = calculate_expected_loss(sample, model_1, scaler, feature_columns)
print("Expected Loss:", el)


XGBoost
Now let’s try training and evaluating a more advanced model (XGBoost), then compare it to the logistic regression model.

In [ ]:
conda install -c conda-forge xgboost -y

In [ ]:
import xgboost as xgb

In [ ]:
# Train XGBoost classifier
model_2 = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
model_2.fit(X_train, y_train)

# Predict probabilities and classes
y_proba_xgb = model_2.predict_proba(X_test)[:, 1]
y_pred_xgb = model_2.predict(X_test)
print("ROC AUC Score:", roc_auc_score(y_test, y_proba_xgb))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_xgb)
plt.figure(figsize=(6,5), dpi=300)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Default', 'Default'],
            yticklabels=['Non-Default', 'Default'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()